In [1]:
from pyspark.sql import SparkSession
import pyspark

AWS_ACCESS_KEY = "minioadmin"
AWS_SECRET_KEY = "minioadmin"
AWS_S3_ENDPOINT = "http://minio_server:9000"
WAREHOUSE = "s3a://gold/" 
NESSIE_URI = "http://nessie:19120/api/v1"

conf = (
    pyspark.SparkConf()
    .setAppName("TrainModel2")  
    .set('spark.jars.packages',
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'org.apache.hadoop:hadoop-aws:3.3.4,'
         'com.amazonaws:aws-java-sdk-bundle:1.12.300')
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.authentication.type", "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint", AWS_S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.access-key", AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-key", AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
)

spark = (
    SparkSession.builder
    .config(conf=conf) 
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")


In [2]:
df_fact = spark.table("nessie.gold_db.fact_order")
df_customer = spark.table("nessie.gold_db.dim_customer")
df_product = spark.table("nessie.gold_db.dim_product")
df_time = spark.table("nessie.gold_db.dim_time")
df_location = spark.table("nessie.gold_db.dim_location")

In [3]:
query = """
SELECT 
    f.time_id,
    f.customer_id,
    f.product_id,
    f.location_id,
    f.quantity,
    f.unit_price,        
    f.total_revenue,     

    -- Dim_time
    t.order_date,
    t.year,
    t.month,
    t.day,
    t.quarter,
    t.weekday_name,

    -- Dim_customer
    c.gender,
    c.age_group,
    c.education,
    c.income,
    c.race,
    c.state,             -- Cột state từ bảng khách hàng

    -- Dim_product
    p.product_title,
    p.product_category,

    -- Dim_location
    l.state_name purchase_state,
    l.region purchase_region

FROM nessie.gold_db.fact_order f
LEFT JOIN nessie.gold_db.dim_time t     ON f.time_id     = t.time_id
LEFT JOIN nessie.gold_db.dim_customer c ON f.customer_id  = c.customer_id
LEFT JOIN nessie.gold_db.dim_product p   ON f.product_id  = p.product_id
LEFT JOIN nessie.gold_db.dim_location l  ON f.location_id = l.location_id
"""

In [4]:
df_fact_full = spark.sql(query)
df_fact_full.limit(10).toPandas()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

,time_id,customer_id,product_id,location_id,quantity,unit_price,total_revenue,order_date,year,month,...,gender,age_group,education,income,race,state,product_title,product_category,purchase_state,purchase_region
0,20221126,R_262vJPpUeeFKuHT,0307118932,34359738369,1.0,3.99,3.99,2022-11-26,2022,11,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Just Grandma and Me (Little Critter) (Pictureb...,ABIS_BOOK,Arizona,West
1,20210419,R_2AWkL1j4mAcAll8,0307464970,34359738369,1.0,15.99,15.99,2021-04-19,2021,4,...,Male,18 - 24 years,High school diploma or GED,"$25,000 - $49,999",American Indian/Native American or Alaska Native,Arizona,The Harlem Hellfighters,ABIS_BOOK,Arizona,West
2,20200609,R_tMsVLA8C6RuTmlb,048640708X,34359738369,1.0,7.89,7.89,2020-06-09,2020,6,...,Female,55 - 64 years,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Chinese Painting Techniques (Dover Art Instruc...,ABIS_BOOK,Arizona,West
3,20191221,R_262vJPpUeeFKuHT,0812971833,34359738369,1.0,12.60,12.60,2019-12-21,2019,12,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Olive Kitteridge,ABIS_BOOK,Arizona,West
4,20221214,R_AmMCFCEBSSiRhLz,1492677698,34359738369,1.0,17.99,17.99,2022-12-14,2022,12,...,Female,35 - 44 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Arizona,The Complete Baking Book for Young Chefs: 100+...,ABIS_BOOK,Arizona,West
5,20200826,R_3G9b1ToPNT28Utq,1627797645,34359738369,1.0,6.50,6.50,2020-08-26,2020,8,...,Female,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Arizona,Marlena: A Novel,ABIS_BOOK,Arizona,West
6,20221004,R_262vJPpUeeFKuHT,1637315309,34359738369,1.0,11.99,11.99,2022-10-04,2022,10,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Hooty Pooty the Owl: A Funny Rhyming Halloween...,ABIS_BOOK,Arizona,West
7,20190401,R_2E0Q3pbtzJQqLg7,B00004R9TL,34359738369,1.0,20.85,20.85,2019-04-01,2019,4,...,Male,35 - 44 years,Bachelor's degree,"$25,000 - $49,999",White or Caucasian,Florida,"BLACK+DECKER Trimmer Line, 30-Foot, 0.065-Inch...",IGNITION_COIL,Arizona,West
8,20191127,R_06RZP9pS7kONINr,B00006ANDK,996432412672,1.0,19.99,19.99,2019-11-27,2019,11,...,Female,65 and older,Bachelor's degree,"$75,000 - $99,999",White or Caucasian,South Dakota,Oral-B Sensitive Gum Care Electric Toothbrush ...,TOOTHBRUSH_HEAD,South Dakota,Midwest
9,20210118,R_4PC8xUs7y9n494R,B0000BYEF6,34359738369,2.0,12.34,24.68,2021-01-18,2021,1,...,Female,25 - 34 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Arizona,Lutron Credenza Plug-In Dimmer for Incandescen...,ELECTRONIC_SWITCH,Arizona,West


In [15]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator

# =====================================================
# 1. FEATURE ENGINEERING NÂNG CAO
# =====================================================
# Gom nhóm dữ liệu theo khách hàng và lấy thêm các thông tin hành vi/nhân khẩu học
df_enhanced = (
    df_fact_full.groupBy("customer_id")
    .agg(
        F.sum("total_revenue").alias("total_spend"),
        F.countDistinct("time_id").alias("total_orders"),
        F.countDistinct("product_category").alias("category_diversity"),
        F.avg("quantity").alias("avg_items_per_line"),
        F.min("year").alias("first_year"),
        F.max("year").alias("last_year"),
        # Lấy thông tin định tính (Categorical)
        F.first("gender").alias("gender"),
        F.first("income").alias("income")
    )
    .withColumn("years_active", (F.col("last_year") - F.col("first_year") + 1))
    .withColumn("years_active", F.when(F.col("years_active") <= 0, 1).otherwise(F.col("years_active")))
    
    # TARGET: Tần suất đơn hàng trung bình mỗi năm
    .withColumn("avg_orders_per_year", F.col("total_orders") / F.col("years_active"))
    
    # Xử lý giá trị trống cho Pipeline
    .na.fill("Unknown", ["gender", "income"])
    .na.fill(0)
)

# =====================================================
# 2. TIỀN XỬ LÝ (PRE-PROCESSING PIPELINE)
# =====================================================
# Định nghĩa các nhóm cột
cat_cols = ["gender", "income"]
num_cols = ["total_spend", "category_diversity", "avg_items_per_line", "years_active"]

# Chuyển String thành Index -> OneHot (Dành cho Random Forest/XGBoost)
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") for c in cat_cols]

# Gom tất cả vào Vector Features
assembler = VectorAssembler(
    inputCols=num_cols + [f"{c}_vec" for c in cat_cols],
    outputCol="rawFeatures"
)

# Chuẩn hóa dữ liệu (Scaling)
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withMean=True, withStd=True)


In [16]:
# =====================================================
# 3. MÔ HÌNH VÀ TUNING
# =====================================================
# Định nghĩa Model
rf = RandomForestRegressor(
    labelCol="avg_orders_per_year", 
    featuresCol="features",
    seed=42
)

# Xây dựng Pipeline tổng thể
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

# Chia dữ liệu Train/Test
train_df, test_df = df_enhanced.randomSplit([0.8, 0.2], seed=42)

# Thiết lập Grid tìm tham số tốt nhất
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [200, 300 ,400])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)

evaluator = RegressionEvaluator(labelCol="avg_orders_per_year", metricName="r2")

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=4
)
# 4. HUẤN LUYỆN 
# =====================================================
print("🏗️ Đang huấn luyện mô hình dự đoán Tần suất đơn hàng...")
# tvs.fit sẽ thực hiện cross-validation và trả về model tốt nhất
rf_tvs_model = tvs.fit(train_df)


🏗️ Đang huấn luyện mô hình dự đoán Tần suất đơn hàng...


In [17]:
# ĐÁNH GIÁ
# Sử dụng bestModel để dự đoán trên cả 2 tập dữ liệu
train_pred = rf_tvs_model.bestModel.transform(train_df)
test_pred  = rf_tvs_model.bestModel.transform(test_df)

# Thiết lập target (đã định nghĩa ở bước trước là avg_orders_per_year)
target = "avg_orders_per_year"

print("\n" + "="*30)
print("📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH")
print("="*30)

# --------------------------
# Vòng lặp tính toán các chỉ số (Metrics)
# --------------------------
metrics = ['r2', 'mae', 'rmse']

for metric in metrics:
    # Khởi tạo evaluator cho từng chỉ số
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    
    # Tính toán kết quả cho tập Train và Test
    train_score = evaluator.evaluate(train_pred)
    test_score  = evaluator.evaluate(test_pred)
    
    # In kết quả theo format Thăng yêu cầu
    print(f"{metric.upper()} Train:", round(train_score, 4))
    print(f"{metric.upper()} Test :", round(test_score, 4))


📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH
R2 Train: 0.887
R2 Test : 0.8291
MAE Train: 6.1544
MAE Test : 7.1817
RMSE Train: 9.3948
RMSE Test : 12.5259


In [18]:
# 1. Lấy mô hình tốt nhất
best_pipeline = rf_tvs_model.bestModel
rf_model = best_pipeline.stages[-1]

# 2. Tìm stage VectorAssembler để lấy tên các cột đầu vào
assembler_stage = [s for s in best_pipeline.stages if hasattr(s, "getInputCols")][-1]
container = assembler_stage.getInputCols()

# 3. Lấy mảng giá trị quan trọng
importances = rf_model.featureImportances.toArray()

print("="*45)
print("⭐ BẢNG XẾP HẠNG ĐỘ QUAN TRỌNG FEATURES")
print("="*45)

# Gom tên và giá trị vào list để sắp xếp
feat_imp = []
for i, name in enumerate(container):
    # Một số cột Vector (như OneHot) có thể chiếm nhiều chỉ số trong importances
    # Nhưng ở mức cơ bản, chúng ta ánh xạ 1-1 với đầu vào của Assembler
    if i < len(importances):
        feat_imp.append((name, round(float(importances[i]) * 100, 2)))

# Sắp xếp giảm dần theo %
feat_imp.sort(key=lambda x: x[1], reverse=True)

for name, val in feat_imp:
    print(f"{name:<25}: {val:>6}%")

print("="*45)

⭐ BẢNG XẾP HẠNG ĐỘ QUAN TRỌNG FEATURES
category_diversity       :  50.33%
total_spend              :   39.4%
avg_items_per_line       :   4.84%
years_active             :   2.62%
gender_vec               :   0.34%
income_vec               :   0.32%
